# TP cœur réseau — Convergence distribuée et boucle transitoire

Objectif : distinguer un calcul final de plus court chemin de l'installation
asynchrone des tables de transfert. Topologie : A–B coût 1, B–C coût 1, A–C coût 3.
Destination C. B–C tombe à t=0. Dans le premier cas B installe sa nouvelle FIB
à 5 ms et A à 30 ms ; dans le second, l'ordre est inversé.

Il s'agit d'un modèle d'événements et de FIB, **pas** d'une implémentation OSPF,
d'un démon FRRouting ni d'un paquet réel. La détection et les délais sont imposés.
Aucun réseau ni équipement n'est configuré. Les liens du modèle sont bidirectionnels.

Prérequis : Python 3.10+, pandas/Jupyter dans l'environnement verrouillé
`machine_learning_labs`. Exécuter les notebooks depuis le livre ou un sous-dossier.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

book_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "core_network_labs" / "inputs" / "scenarios.json").is_file()
    ),
    None,
)
if book_root is None:
    raise RuntimeError(
        "Ouvrir ce notebook depuis le dossier du livre ou un de ses sous-dossiers."
    )
if str(book_root) not in sys.path:
    sys.path.insert(0, str(book_root))
from core_network_labs.run_labs import build_results  # noqa: E402 - après résolution du dossier du livre

In [ ]:
results = build_results()["convergence.json"]["cases"]
summary = pd.DataFrame(
    [
        {
            "order": order,
            "time_ms": row["time_ms"],
            "status": row["status"],
            "path": " → ".join(row["path"]),
            "FIB": row["fib"],
        }
        for order, rows in results.items()
        for row in rows
    ]
)
summary

In [ ]:
unordered = {row["time_ms"]: row for row in results["unordered"]}
assert unordered[-1]["path"] == ["A", "B", "C"]
assert unordered[0]["status"] == "failed_link"
assert unordered[5]["path"] == ["A", "B", "A"]
assert unordered[30]["path"] == ["A", "C"]
assert not any(row["status"] == "forwarding_loop" for row in results["ordered"])
summary[summary.time_ms.isin([5, 30])]

## Questions et résultat attendu

À 5 ms, A utilise encore B tandis que B utilise A : la route finale est sans
boucle, mais le transfert instantané boucle. Avant cette installation, le
problème est différent : l'ancienne FIB vise le lien en panne. À 30 ms les
deux cas livrent via A–C. Expliquer pourquoi « plus de boucle » ne signifie
pas nécessairement absence de perte ni rétablissement complet du service.

Contre-épreuve : modifier **en mémoire** les instants d'installation et
prédire les chemins avant de relancer `convergence_case`. Ne pas généraliser
l'ordre favorable de ce triangle à une garantie pour toute topologie.

Sources : RFC 5715 (micro-boucles), RFC 5286 (réparation locale et limites).
Entrées : `core_network_labs/inputs/scenarios.json` ; preuve attendue :
`core_network_labs/results/convergence.json`. Redémarrer le noyau réinitialise
entièrement l'expérience.
